# 예제 04. LSTM 시계열 예측
빅데이터프로그래밍 · 11주차

## 목표
- 시계열 데이터 생성 → Dataset 구성 → LSTM 작성 → 예측
- 실제 값과 예측 값을 선 그래프로 겹쳐 그린다
- 예측 오차를 확인한다

5주차 학습 루프가 그대로 쓰입니다. 데이터와 모델만 시계열용으로 바뀝니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(42)
np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 시계열 데이터 생성


In [ ]:
T = 1000
t = np.arange(T)
series = np.sin(t * 0.05) + 0.3 * np.sin(t * 0.2) + np.random.randn(T) * 0.05

plt.figure(figsize=(12, 3))
plt.plot(series, linewidth=1)
plt.title("두 주기 + 잡음"); plt.grid(alpha=.3)
plt.show()


## 2. Dataset 구성


In [ ]:
SEQ_LEN = 40

class SeriesDataset(Dataset):
    def __init__(self, arr, seq_len):
        self.arr = torch.tensor(arr, dtype=torch.float32)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.arr) - self.seq_len
    def __getitem__(self, i):
        return (self.arr[i:i+self.seq_len].unsqueeze(-1),
                self.arr[i+self.seq_len].unsqueeze(-1))


n_train = int(T * 0.8)
train_series, val_series = series[:n_train], series[n_train:]

train_ds = SeriesDataset(train_series, SEQ_LEN)
val_ds   = SeriesDataset(val_series,   SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

print(f"학습 표본 {len(train_ds)}개 · 검증 표본 {len(val_ds)}개")
x, y = train_ds[0]
print("x:", tuple(x.shape), "y:", tuple(y.shape))


## 3. LSTM 모델 작성


In [ ]:
class LSTMForecast(nn.Module):
    def __init__(self, hidden=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)          # (batch, 시점, hidden)
        return self.fc(out[:, -1, :])  # 마지막 시점으로 예측


model = LSTMForecast().to(device)
print(model)
print("파라미터:", f"{sum(p.numel() for p in model.parameters()):,}개")
print("출력:", tuple(model(x.unsqueeze(0).to(device)).shape))


## 4. 학습
회귀 문제이므로 손실은 MSE입니다. 분류의 CrossEntropyLoss가 아닙니다.


In [ ]:
loss_fn = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    total = n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            total += loss_fn(model(xb), yb).item() * len(yb)
            n += len(yb)
    return total / n


history = []
for epoch in range(1, 41):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    history.append((evaluate(train_loader), evaluate(val_loader)))
    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch {epoch:2d}  학습 {history[-1][0]:.5f}  검증 {history[-1][1]:.5f}")


In [ ]:
xs = range(1, len(history)+1)
plt.figure(figsize=(8, 3.4))
plt.plot(xs, [h[0] for h in history], label="학습")
plt.plot(xs, [h[1] for h in history], label="검증")
plt.title("MSE loss"); plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=.3)
plt.show()


## 5. 예측 결과 시각화 — 실제 값과 겹쳐 그리기


In [ ]:
model.eval()
preds, trues = [], []
with torch.no_grad():
    for xb, yb in val_loader:                # shuffle=False 라 순서가 유지됩니다
        preds.append(model(xb.to(device)).cpu())
        trues.append(yb)

preds = torch.cat(preds).squeeze().numpy()
trues = torch.cat(trues).squeeze().numpy()

plt.figure(figsize=(12, 4))
plt.plot(trues, label="실제", linewidth=1.6)
plt.plot(preds, label="예측", linewidth=1.4, linestyle="--")
plt.title("검증 구간 — 실제 값과 예측 값"); plt.legend(); plt.grid(alpha=.3)
plt.show()


## 6. 예측 오차 확인


In [ ]:
err = preds - trues

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].plot(err, linewidth=1); ax[0].axhline(0, color="crimson", linestyle="--")
ax[0].set_title("오차 (예측 − 실제)"); ax[0].grid(alpha=.3)
ax[1].hist(err, bins=30); ax[1].set_title("오차 분포")
ax[2].scatter(trues, preds, s=6, alpha=.5)
lims = [min(trues.min(), preds.min()), max(trues.max(), preds.max())]
ax[2].plot(lims, lims, color="crimson", linestyle="--")
ax[2].set_xlabel("실제"); ax[2].set_ylabel("예측"); ax[2].set_title("실제 vs 예측")
plt.tight_layout(); plt.show()


In [ ]:
mae = np.abs(err).mean()
rmse = np.sqrt((err**2).mean())
print(f"MAE  {mae:.5f}")
print(f"RMSE {rmse:.5f}")
print(f"실제 값 범위 {trues.min():.3f} ~ {trues.max():.3f}")
print(f"→ 범위 대비 RMSE {rmse/(trues.max()-trues.min())*100:.2f}%")


## 7. 학습 구간까지 함께 그려 보기


In [ ]:
model.eval()
all_ds = SeriesDataset(series, SEQ_LEN)
all_loader = DataLoader(all_ds, batch_size=64, shuffle=False)

all_pred = []
with torch.no_grad():
    for xb, _ in all_loader:
        all_pred.append(model(xb.to(device)).cpu())
all_pred = torch.cat(all_pred).squeeze().numpy()

plt.figure(figsize=(13, 4))
plt.plot(range(SEQ_LEN, T), series[SEQ_LEN:], label="실제", linewidth=1.2)
plt.plot(range(SEQ_LEN, T), all_pred, label="예측", linewidth=1, linestyle="--")
plt.axvline(n_train, color="crimson", linestyle=":", label="학습/검증 경계")
plt.legend(); plt.grid(alpha=.3); plt.title("전체 구간")
plt.show()


## 8. 여러 시점을 이어서 예측하기
예측값을 다시 입력으로 넣습니다. 오차가 누적되어 점점 벌어집니다.


In [ ]:
model.eval()
seed = torch.tensor(series[n_train-SEQ_LEN:n_train], dtype=torch.float32)
window = seed.clone()
rollout = []

with torch.no_grad():
    for _ in range(100):
        p = model(window.unsqueeze(0).unsqueeze(-1).to(device)).cpu().squeeze()
        rollout.append(p.item())
        window = torch.cat([window[1:], p.reshape(1)])

plt.figure(figsize=(12, 3.6))
plt.plot(range(100), series[n_train:n_train+100], label="실제", linewidth=1.6)
plt.plot(range(100), rollout, label="이어서 예측", linewidth=1.4, linestyle="--")
plt.legend(); plt.grid(alpha=.3); plt.title("100 시점 연속 예측 — 오차가 누적됩니다")
plt.show()


In [ ]:
torch.save(model.state_dict(), "lstm_forecast.pt")
print("저장 완료")


## 직접 해보기
1. `SEQ_LEN` 을 10으로 줄이면 예측이 어떻게 되나요?
2. 잡음을 0.3으로 키우면 RMSE가 얼마나 올라가나요?
3. `hidden` 을 16으로 줄이면 어떻게 되나요?


In [ ]:
# 여기에 작성하세요
